In [1]:
import os
import csv
import time
import pycdlib

In [2]:
# ---------- Configuration ----------

SOURCE_DIR = r"G:\thecall\done"
DEST_DIR   = r"e:\c_out"
CSV_PATH   = os.path.join(DEST_DIR, "inventory.csv")
ERROR_LOG  = os.path.join(DEST_DIR, "errors.log")

MAX_FILENAME_LEN = 200  # truncate filenames longer than this (before extension)


In [3]:
# ---------- Utilities ----------

def safe_path(path):
    """Prepend the Windows long-path prefix so paths > 260 chars work."""
    path = os.path.abspath(path)
    if not path.startswith("\\\\?\\"):
        return "\\\\?\\" + path
    return path


def clean_name(name):
    """Remove characters illegal on Windows and strip ISO version suffix (;1).

    Windows silently rejects directory/file names that end with a space or
    period even when using the long-path prefix, so we rstrip both.
    """
    if ";" in name:
        name = name[:name.rfind(";")]
    name = "".join(c if c not in '<>:"/\\|?*' else "_" for c in name)
    return name.rstrip(" .")


def truncate_name(name, max_len=MAX_FILENAME_LEN):
    """Truncate a filename while preserving its extension."""
    root, ext = os.path.splitext(name)
    if len(name) <= max_len:
        return name
    return root[: max_len - len(ext)] + ext


def detect_path_type(iso):
    """
    Choose the best path namespace for this ISO:
      'joliet_path' - disc has Joliet extensions (long, Unicode-friendly names)
      'iso_path'    - plain ISO 9660 (8.3 or slightly relaxed names)
    Rock Ridge is skipped; these discs don't have it.
    """
    if iso.joliet_vd is not None:
        return "joliet_path"
    return "iso_path"

In [4]:
# ---------- Date extraction (Final Refinement) ----------

def _parse_dir_record_date(dt):
    """Parse a DirectoryRecordDate (7-byte format)."""
    try:
        year = getattr(dt, "years_since_1900", None)
        day = getattr(dt, "day_of_month", None)
        if year is None or year == 0 or day is None:
            return ""
        return "{:04d}-{:02d}-{:02d} {:02d}:{:02d}:{:02d}".format(
            year + 1900, dt.month, day, dt.hour, dt.minute, dt.second,
        )
    except: return ""

def _parse_vd_date(dt):
    """Parse a VolumeDescriptorDate (17-byte ASCII format)."""
    if dt is None: return ""
    dstr = getattr(dt, "date_str", b"").decode("ascii", errors="ignore").strip("\x00 ")
    # Check if we have at least YYYYMMDD
    if len(dstr) >= 8 and dstr[:4].isdigit() and dstr[:4] != "0000":
        try:
            return "{}-{}-{} {}:{}:{}".format(
                dstr[0:4], dstr[4:6], dstr[6:8],
                dstr[8:10] or "00", dstr[10:12] or "00", dstr[12:14] or "00"
            )
        except: pass
    return ""

def get_all_dates(iso, iso_file_path, path_type):
    results = {"date_primary": "", "date_secondary": "", "date_volume": "", "date_source": ""}
    
    # 1. File Date (Primary Namespace)
    try:
        rec = iso.get_record(**{path_type: iso_file_path})
        results["date_primary"] = _parse_dir_record_date(rec.date)
    except: pass

    # 2. Volume Dates (The "Disc Date" - checking all 4 standard fields)
    pvd = iso.pvd
    v_cre = _parse_vd_date(getattr(pvd, "cre_date", None))
    v_mod = _parse_vd_date(getattr(pvd, "mod_date", None))
    v_eff = _parse_vd_date(getattr(pvd, "eff_date", None))
    
    # Use the first valid date found for the volume
    results["date_volume"] = v_mod or v_cre or v_eff
    
    # Label the source
    if results["date_primary"]:
        results["date_source"] = path_type
    elif results["date_volume"]:
        results["date_source"] = "volume_descriptor"
        
    return results

def _date_to_epoch(date_str):
    if not date_str: return None
    try: return time.mktime(time.strptime(date_str, "%Y-%m-%d %H:%M:%S"))
    except: return None

def stamp_file_date(out_path, dates):
    """Apply the best available date to the physical file on disk."""
    best = dates["date_primary"] or dates["date_volume"]
    epoch = _date_to_epoch(best)
    if epoch:
        try: os.utime(os.path.abspath(out_path), (epoch, epoch))
        except: pass

def log_error(message):
    with open(ERROR_LOG, "a", encoding="utf-8") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {message}\n")

In [5]:
# ---------- Extraction Logic ----------
def extract_iso(iso_path, output_dir, writer, parent_label):
    
    iso_name = os.path.basename(iso_path)
    iso = pycdlib.PyCdlib()
    try:
        iso.open(iso_path)
    except Exception as e:
        log_error(f"OPEN_FAIL | {iso_path} | {str(e)}")
        return

    path_type = detect_path_type(iso)
    print(f"  Path namespace: {path_type}")

    for dir_path, dirs, files in iso.walk(**{path_type: "/"}):
        for file in files:
            iso_file_path = dir_path.rstrip("/") + "/" + file
            
            try:
                clean_file = truncate_name(clean_name(file))
                rel_dir = dir_path.lstrip("/")
                if rel_dir:
                    rel_dir = os.path.join(*[clean_name(p) for p in rel_dir.split("/") if p])
                else:
                    rel_dir = "."

                out_dir = os.path.join(output_dir, rel_dir)
                out_path = os.path.join(out_dir, clean_file)

                os.makedirs(safe_path(out_dir), exist_ok=True)
                iso.get_file_from_iso(
                    local_path=safe_path(out_path),
                    **{path_type: iso_file_path},
                )

                dates = get_all_dates(iso, iso_file_path, path_type)
                stamp_file_date(out_path, dates)

                writer.writerow({
                    "iso_file":       iso_name,
                    "parent_folder":  parent_label,
                    "file_name":      clean_file,
                    "original_name":  file,
                    "folder":         rel_dir,
                    "full_path":      out_path,
                    "date_primary":   dates["date_primary"],
                    "date_secondary": dates["date_secondary"],
                    "date_volume":    dates["date_volume"],
                    "date_source":    dates["date_source"],
                    "path_type":      path_type,
                })
            except Exception as e:
                log_error(f"FILE_ERROR | {iso_name} | {iso_file_path} | {str(e)}")

    iso.close()

In [8]:
def main():
    os.makedirs(DEST_DIR, exist_ok=True)
    
    fieldnames = [
        "iso_file", "parent_folder", "file_name", "original_name", # Added parent_folder
        "folder", "full_path",
        "date_primary", "date_secondary", "date_volume",
        "date_source", "path_type",
    ]

    with open(CSV_PATH, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for root, _, files in os.walk(SOURCE_DIR):
            # Capture the numeric label from the folder name
            parent_folder_name = os.path.basename(root) 
            
            for file in files:
                if file.lower().endswith(".iso"):
                    iso_path = os.path.join(root, file)
                    iso_name_no_ext = os.path.splitext(file)[0]
                    
                    # New Naming: {Number}_{ISO_Name}
                    new_folder_name = f"{parent_folder_name}_{iso_name_no_ext}"
                    out_folder = os.path.join(DEST_DIR, clean_name(new_folder_name))
                    
                    os.makedirs(safe_path(out_folder), exist_ok=True)
                    print(f"Processing: {iso_path} -> {new_folder_name}")

                    try:
                        # Pass the parent_folder_name into the extraction logic
                        extract_iso(iso_path, out_folder, writer, parent_folder_name)
                    except Exception as e:
                        log_error(f"FATAL | {iso_path} | {str(e)}")

In [9]:

if __name__ == "__main__":
    main()


Processing: G:\thecall\done\12\Call 07-25-08-01.iso -> 12_Call 07-25-08-01
  Path namespace: joliet_path
Processing: G:\thecall\done\13\Call 08-08-08-15.iso -> 13_Call 08-08-08-15
  Path namespace: joliet_path
Processing: G:\thecall\done\14\Call 08-22-08-29.iso -> 14_Call 08-22-08-29
  Path namespace: joliet_path
Processing: G:\thecall\done\15\Call 09-05.iso -> 15_Call 09-05
  Path namespace: joliet_path
Processing: G:\thecall\done\16\09-13 Call.iso -> 16_09-13 Call
  Path namespace: joliet_path
Processing: G:\thecall\done\17\10-03 Call.iso -> 17_10-03 Call
  Path namespace: joliet_path
Processing: G:\thecall\done\1\05-02 Call.iso -> 1_05-02 Call
  Path namespace: joliet_path
Processing: G:\thecall\done\2\CALL 04-18-04-25.ISO -> 2_CALL 04-18-04-25
  Path namespace: joliet_path
Processing: G:\thecall\done\3\CALL 04-04-04-11.ISO -> 3_CALL 04-04-04-11
  Path namespace: joliet_path
Processing: G:\thecall\done\4\Call 05-09.iso -> 4_Call 05-09
  Path namespace: joliet_path
Processing: G:\the

KeyboardInterrupt: 